# Company research: one CSV, one notebook

Put this notebook beside **Company_Research_Master.csv.gz**. All views below use only that file. It bundles existing financial downloads, target/candidate context, repaired facets, scores, conflict records and cached source documents. It does not rerun downloads, global discovery, factual verification or forecasting.

**Important:** 2,111 flagged fields remain unresolved and excluded from scoring. Automatic excerpts and retained baselines are not individually verified. Scores rerank the existing ten candidates only. Target selector evidence is official website only; external target records and financials are diagnostic/held out. Source dates and review dates remain separate.

The CSV is a long-format record store: `dataset` identifies the table, and `record_json` preserves its complete fields. Different record types occupy separate rows, avoiding many-to-many joins that multiply observations. Embedded source documents make the notebook independent of old local source paths. Publication links and original snapshot paths are retained for traceability. Large raw documents make chunked loading preferable.

In [ ]:
from pathlib import Path
import csv, json, collections, gzip
import pandas as pd
csv.field_size_limit(2**31 - 1)
DATA = Path('Company_Research_Master.csv.gz')
assert DATA.exists(), 'Place the master CSV beside this notebook.'

def records(dataset):
    with gzip.open(DATA, 'rt', encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            if row['dataset'] == dataset:
                item = json.loads(row['record_json'])
                if row['record_type'] == 'raw_source_document':
                    item['Source Text'] = row['source_text']
                yield item

def table(dataset):
    return pd.DataFrame.from_records(records(dataset))

manifest = next(records('Delivery_Manifest'))
pd.Series(manifest['Dataset Counts'], name='rows').sort_index()

## Updated comparables and facet contributions
Scores withheld for missing core evidence remain missing. Inspect coverage alongside similarity; missing evidence is not a zero similarity.

In [ ]:
comps = table('Corrected_Comp_Results')
facets = table('Corrected_Facets')
contributions = table('Corrected_Facet_Scores')
dispositions = table('Conflict_Dispositions')
assert len(comps) == 5490
assert len(facets) == 113310
assert len(contributions) == 87840
assert len(dispositions) == 3016
assert not facets.duplicated(['Entity Role','Entity ID','Facet']).any()
quarantined = facets['Repair Action'].eq('UNRESOLVED_VALUE_QUARANTINED')
assert facets.loc[quarantined, 'Corrected Value'].isna().all()
assert not facets.loc[quarantined, 'Eligible For Scoring'].any()
assert facets.loc[facets['Entity Role'].eq('TARGET') & facets['Eligible For Scoring'], 'Source Type'].eq('OFFICIAL_WEBSITE').all()
assert dispositions['Repair Action'].eq('UNRESOLVED_VALUE_QUARANTINED').sum() == 2111
comps.head(10)

In [ ]:
# Change the search text for any target.
query = 'Paylocity'
comps[comps['Target Name'].str.contains(query, case=False, na=False)]

## Before/after, sources and unresolved evidence

In [ ]:
facets[facets['Company Name'].str.contains(query, case=False, na=False)][['Entity Role','Company Name','Facet','Before Value','Corrected Value','Eligible For Scoring','Source Link','Before Source Link','Repair Action','Human Reviewed Fact']]

In [ ]:
target_sources = table('Target_Sources')
public_sources = table('Public_Sources')
# External target references are diagnostic; do not merge them into selector inputs.
public_sources.head(3)

## Financial data and FF49 context
These are the existing snapshots, with their original identifiers, units, dates and coverage limitations. Presence of a dataset does not establish complete coverage or point-in-time validity. Target financials are not permitted selector inputs under the current pseudo-private policy.

In [ ]:
available = manifest['Dataset Counts']
pd.Series({k:v for k,v in available.items() if k.startswith(('Prepared_', 'LSEG_'))}, name='rows')

In [ ]:
# Load only the financial table you need; all original columns are retained.
annual_financials = table('LSEG_annual_five_years')
ff49_benchmarks = table('Prepared_FF49_Benchmarks_560')
annual_financials.head()

## Read an embedded raw source document
Original local snapshot paths are provenance references. The text itself is embedded in the master CSV, so those paths need not exist on another computer. To save memory this example reads one document rather than loading every full text.

In [ ]:
doc = next(records('Embedded_Source_Documents'), None)
if doc:
    print(doc['Source Link'])
    print(doc['Source Text'][:6000])
print('Missing cached source paths:', len(manifest['Missing Cached Source Paths']))

## Scoring reconciliation
Retains original proposed weights and checks observed-facet weighted averages. This verifies arithmetic, not the factual accuracy of automatically extracted claims or the quality of the candidate pool.

In [ ]:
keys = ['Target ID','Candidate RIC']
s = contributions.copy()
s['observed_weight'] = s['Proposed Weight'].where(s['Facet Similarity Percent'].notna(), 0)
s['numerator'] = s['observed_weight'] * s['Facet Similarity Percent'].fillna(0)
check = s.groupby(keys)[['observed_weight','numerator']].sum()
check['recomputed'] = check['numerator'].div(check['observed_weight'].replace(0, float('nan')))
check = check.join(comps.set_index(keys)[['Partial Observed-Facet Similarity','Weighted Evidence Coverage']])
assert (check['observed_weight'] - check['Weighted Evidence Coverage']).abs().max() < 1e-4
assert (check['recomputed'] - check['Partial Observed-Facet Similarity']).abs().max() < .011
print('Structural, policy and score arithmetic checks passed. Factual conflicts remain explicitly unresolved.')